# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## 00. Imports

In [1]:
from loader import CustomIAPRDataloader
from models.cnn import SimpleCNN
from helper import get_device
from trainer import Trainer
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn


device = get_device()
print(f"Using device: {device}")

Using device: mps


## 01. Load the dataset

In [2]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),  # You can choose a suitable resolution
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Standard ImageNet normalization
    #                     std=[0.229, 0.224, 0.225]),
])

DATASET_DIR = "dataset_project_iapr2025"
loader = CustomIAPRDataloader(base_dir=DATASET_DIR, transform=transform)

full_train_ds = loader.train_dataset 
test_ds = loader.test_dataset
ref_ds = loader.reference_dataset
class_number = loader.class_number
class_names = loader.class_names

train_size = int(0.8 * len(full_train_ds))
val_size = len(full_train_ds) - train_size

train_ds, val_ds = random_split(full_train_ds, [train_size, val_size])

# Display information about the datasets
print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")
print(f"Test dataset size: {len(test_ds)}")
print(f"Reference dataset size: {len(ref_ds)}")

# Create DataLoader for training and testing datasets
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

# Example of how to use the DataLoader
for images, labels in train_loader:

    print(f"Batch size: {images.size(0)}")
    print(f"Image shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    break  # Remove this to iterate through the entire dataset

# display first image from the train dataset
import matplotlib.pyplot as plt
import numpy as np
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# imshow(train_ds[0][0])

Train dataset size: 72
Validation dataset size: 18
Test dataset size: 180
Reference dataset size: 13
Batch size: 8
Image shape: torch.Size([8, 3, 512, 512])
Labels shape: torch.Size([8, 13])


## 02. Create CNN model
Train, Evaluate and Predict

In [ ]:
model = SimpleCNN(input_shape=3, hidden_units=64, image_height=512, image_width=512, output_shape=class_number)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

trainer = Trainer(model=model,
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      num_epochs=10,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
predictions = trainer.predict()
print(predictions)
trainer.save_model()
avg_loss, accuracy = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

Epoch [1/10], Loss: 0.7562
Epoch [2/10], Loss: 0.6934
Epoch [3/10], Loss: 0.6937
Epoch [4/10], Loss: 0.6938
Epoch [5/10], Loss: 0.6924
Epoch [6/10], Loss: 0.6913
